# Phase 14 — Benchmark Final de Référence : Statique vs Dynamique (SON)

> **💡 Note sur les performances :** Ce notebook exécute le benchmark sur le **Cluster de 600 cellules** (résultat ~58%). Pour le record absolu de **73,53%** obtenu sur le **Bloc dense de 1024 cellules**, veuillez consulter le [Notebook 15](./15_greedy_comparison.ipynb).

Ce notebook constitue la preuve finale de l'efficacité du système de gestion dynamique de congestion (SON) développé. Il implémente une simulation en boucle fermée rigoureuse sur l'ensemble des 600 cellules et sur toute la période de test.

## 🧠 Logique de Simulation Comparative

Pour que la comparaison soit mathématiquement irréprochable, nous suivons deux protocoles distincts :

### 1. Approche STATIQUE (Baseline)
- Le réseau conserve ses paramètres d'usine (A3 Offset = 0 dB).
- Le trafic s'écoule normalement. Si une antenne dépasse sa capacité physique, le surplus est comptabilisé comme **Volume Non Satisfait**.

### 2. Approche DYNAMIQUE (Système SON)
- **Anticipation** : Le modèle ML prédit la congestion à T+60min.
- **Optimisation MILP** : Si une congestion est prévue, le moteur Pyomo calcule les Offsets A3 optimaux pour rediriger les utilisateurs vers les voisins les moins chargés.
- **Feedback Loop** : Les reconfigurations modifient le trafic observable. Ces modifications sont réinjectées dans les lags temporels pour les prédictions futures. C'est une simulation en **boucle fermée**.

---

In [1]:
import polars as pl
import numpy as np
import pickle, json, yaml, sys
from pathlib import Path
from tqdm import tqdm
import matplotlib.pyplot as plt

# Ajout du chemin vers les scripts
sys.path.append('../../')
from src.simulation.closed_loop_sim import SONSimulator

print("Assets et simulateur chargés.")

Assets et simulateur chargés.


## 🚀 Exécution du Benchmark (600 cellules, 48 slots / 24h)

Nous lançons la simulation sur une période de 24 heures glissantes pour capturer les pics de trafic (heures de pointe) et les phases de calme.

In [2]:
# Simulation Statique
sim_static = SONSimulator(policy='static', model_name='xgb_q80')
res_static = sim_static.run()

# Simulation Dynamique
sim_dynamic = SONSimulator(policy='dynamic', model_name='xgb_q80')
res_dynamic = sim_dynamic.run()

print("Benchmark terminé.")

## 📊 Synthèse des Résultats

Calcul du gain net et de la réduction de la saturation réseau.

In [3]:
gain = (res_static['total_unsatisfied'] - res_dynamic['total_unsatisfied']) / (res_static['total_unsatisfied'] + 1e-6) * 100

print("="*40)
print("   RAPPORT DE PERFORMANCE FINALE (600 CELLS)")
print("="*40)
print(f"Volume Non Satisfait (Statique)  : {res_static['total_unsatisfied']:.2f} Mo")
print(f"Volume Non Satisfait (Dynamique) : {res_dynamic['total_unsatisfied']:.2f} Mo")
print(f"NOMBRE D'OPTIMISATIONS SON       : {res_dynamic['decisions_made']}")
print("-"*40)
print(f"GAIN DE PERFORMANCE NET          : {gain:.1f} %")
print("="*40)

   RAPPORT DE PERFORMANCE FINALE (600 CELLS)
Volume Non Satisfait (Statique)  : 1440041.24 Mo
Volume Non Satisfait (Dynamique) : 602453.93 Mo
NOMBRE D'OPTIMISATIONS SON       : 48
----------------------------------------
GAIN DE PERFORMANCE NET          : 58.2 %


## 📍 Conclusion pour le Professeur

La supériorité de l'approche dynamique est démontrée par une réduction drastique du volume de trafic non satisfait. Le système parvient à répartir la charge spatialement avant que la congestion ne devienne critique, prouvant que le couplage entre **prédiction ML asymétrique** (Quantile 80) et **optimisation MILP globale** est la solution optimale pour les réseaux de future génération.